# 3. LSTM Modell (Deep Learning Algorithmus)

In diesem Notebook trainieren wir ein Bidirectional LSTM Modell für Sentiment-Analyse.


In [2]:
import os
os.environ["PYTHONIOENCODING"] = "utf-8"
import random
import numpy as np
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from keras import Sequential
from keras.layers import TextVectorization, Embedding, LSTM, Dense, Dropout, Bidirectional
from keras.callbacks import EarlyStopping
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# Add src to path
BASE_DIR = Path().resolve()
sys.path.insert(0, str(BASE_DIR / "src"))
from utils_imdb import read_imdb_split, basic_clean, SEED

# Set reproducibility
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")


ModuleNotFoundError: No module named 'tensorflow.python.trackable'

## 3.1 Daten laden und vorbereiten


In [ ]:
DATA_ROOT = BASE_DIR / "data" / "aclImdb"
OUTPUT_DIR = BASE_DIR / "outputs"
MODELS_DIR = OUTPUT_DIR / "models"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Hyperparameters
MAX_DOCS_PER_CLASS = None  # Set to int for faster dev runs
MAX_VOCAB = 40_000
MAX_LEN = 300
EMBED_DIM = 100
BATCH_SIZE = 64
EPOCHS = 6

print("Loading data...")
train = read_imdb_split(DATA_ROOT / "train", max_docs_per_class=MAX_DOCS_PER_CLASS)
test = read_imdb_split(DATA_ROOT / "test", max_docs_per_class=MAX_DOCS_PER_CLASS)

# Preprocess
Xtr = train["text"].apply(basic_clean).tolist()
Xte = test["text"].apply(basic_clean).tolist()
ytr = train["label"].to_numpy(dtype="int64")
yte = test["label"].to_numpy(dtype="int64")

print(f"Train size: {len(Xtr)}")
print(f"Test size: {len(Xte)}")


## 3.2 Text Vectorization


In [ ]:
# Create and adapt vectorizer
vectorizer = TextVectorization(
    max_tokens=MAX_VOCAB,
    output_mode="int",
    output_sequence_length=MAX_LEN
)

print("Adapting vectorizer to training data...")
ds_text = tf.data.Dataset.from_tensor_slices(Xtr).batch(256)
vectorizer.adapt(ds_text)
print(f"Vocabulary size: {vectorizer.vocabulary_size()}")


## 3.3 Modell definieren


In [ ]:
# Build model
model = Sequential([
    vectorizer,
    Embedding(input_dim=MAX_VOCAB, output_dim=EMBED_DIM, input_length=MAX_LEN),
    Bidirectional(LSTM(128)),
    Dropout(0.4),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid"),
])

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()


## 3.4 Training


In [ ]:
# Training with early stopping
es = EarlyStopping(
    monitor="val_accuracy",
    patience=2,
    mode="max",
    restore_best_weights=True
)

print("Training model...")
history = model.fit(
    x=np.array(Xtr, dtype=object),
    y=ytr,
    validation_split=0.15,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[es],
    verbose=1
)


## 3.5 Training History Visualisierung


In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(history.history['accuracy'], label='Train Accuracy', marker='o')
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy', marker='s')
axes[0].set_title('Model Accuracy', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(history.history['loss'], label='Train Loss', marker='o')
axes[1].plot(history.history['val_loss'], label='Val Loss', marker='s')
axes[1].set_title('Model Loss', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "training_history_lstm.png", dpi=300, bbox_inches='tight')
plt.show()


## 3.6 Evaluation


In [ ]:
# Predictions
print("Making predictions...")
proba = model.predict(np.array(Xte, dtype=object), batch_size=BATCH_SIZE).ravel()
pred = (proba >= 0.5).astype(int)

# Metrics
accuracy = accuracy_score(yte, pred)
print(f"\nTest Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(yte, pred, digits=4))
print("\nConfusion Matrix:")
cm = confusion_matrix(yte, pred)
print(cm)


In [ ]:
# Visualize confusion matrix
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'])
ax.set_title('BiLSTM - Confusion Matrix', fontsize=14, fontweight='bold')
ax.set_ylabel('True Label', fontsize=12)
ax.set_xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrix_lstm.png", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Save model
SAVE_DIR = MODELS_DIR / "lstm_imdb_savedmodel"
model.export(str(SAVE_DIR))
print(f"Model saved: {SAVE_DIR}")

# Save predictions
PRED_PATH = OUTPUT_DIR / "preds_lstm.csv"
pd.DataFrame({
    "text": Xte,
    "y_true": yte,
    "y_pred": pred,
    "proba": proba
}).to_csv(PRED_PATH, index=False, encoding="utf-8")
print(f"Predictions saved: {PRED_PATH}")
